In [0]:
from pyspark.sql.functions import (
    col, avg, count, round, rank, when, lit, current_timestamp,
    dense_rank, percentile_approx
)
from pyspark.sql.window import Window

# Read from Silver - Gold always reads from Silver, never Bronze
df_silver = spark.table("healthcare.silver.cms_hrrp_clean")

# Work only with unsuppressed rows for analytics
# Suppressed rows can't contribute meaningful metrics
df_reportable = df_silver.filter(col("is_suppressed") == False)

print(f"Total rows: {df_silver.count()}")
print(f"Reportable rows: {df_reportable.count()}")

In [0]:
# Gold Model 1: Hospital performance summary
# One row per hospital with aggregated metrics across all conditions

hospital_window = Window.orderBy(col("avg_excess_ratio").desc())

df_hospital_summary = df_reportable \
    .groupBy("facility_id", "facility_name", "state") \
    .agg(
        count("measure_name").alias("conditions_reported"),
        round(avg("excess_readmission_ratio"), 4).alias("avg_excess_ratio"),
        round(avg("predicted_readmission_rate"), 4).alias("avg_predicted_rate"),
        round(avg("expected_readmission_rate"), 4).alias("avg_expected_rate"),
        round(avg("number_of_readmissions"), 0).alias("avg_readmissions"),
        round(avg("number_of_discharges"), 0).alias("avg_discharges")
    ) \
    .withColumn("national_rank", dense_rank().over(hospital_window)) \
    .withColumn("penalty_flag", 
        when(col("avg_excess_ratio") > 1.0, lit(True))
        .otherwise(lit(False))) \
    .withColumn("risk_tier",
        when(col("avg_excess_ratio") >= 1.2, lit("High"))
        .when(col("avg_excess_ratio") >= 1.0, lit("Medium"))
        .otherwise(lit("Low"))) \
    .withColumn("_gold_processed_at", current_timestamp())

print(f"Hospitals in summary: {df_hospital_summary.count()}")

# Preview top 10 worst performing hospitals
df_hospital_summary.select(
    "facility_name", "state", "avg_excess_ratio", 
    "risk_tier", "national_rank", "penalty_flag"
).orderBy("national_rank").show(10, truncate=False)

In [0]:
# Gold Model 2: State performance summary
state_window = Window.orderBy(col("state_avg_excess_ratio").desc())

df_state_summary = df_reportable \
    .groupBy("state") \
    .agg(
        count("facility_id").alias("total_records"),
        round(avg("excess_readmission_ratio"), 4).alias("state_avg_excess_ratio"),
        round(avg("predicted_readmission_rate"), 4).alias("state_avg_predicted_rate"),
        round(avg("number_of_readmissions"), 0).alias("state_avg_readmissions")
    ) \
    .withColumn("state_rank", dense_rank().over(state_window)) \
    .withColumn("state_penalty_flag",
        when(col("state_avg_excess_ratio") > 1.0, lit(True))
        .otherwise(lit(False))) \
    .withColumn("_gold_processed_at", current_timestamp())

print(f"States in summary: {df_state_summary.count()}")

# Top 10 worst performing states
df_state_summary.select(
    "state", "state_avg_excess_ratio", 
    "state_rank", "state_penalty_flag",
    "total_records"
).orderBy("state_rank").show(10, truncate=False)

In [0]:
# Gold Model 3: Condition performance summary
# Which medical conditions drive the most readmissions?

condition_window = Window.orderBy(col("condition_avg_excess_ratio").desc())

df_condition_summary = df_reportable \
    .groupBy("measure_name") \
    .agg(
        count("facility_id").alias("hospitals_reporting"),
        round(avg("excess_readmission_ratio"), 4).alias("condition_avg_excess_ratio"),
        round(avg("predicted_readmission_rate"), 4).alias("condition_avg_predicted_rate"),
        round(avg("number_of_readmissions"), 0).alias("condition_avg_readmissions"),
        round(avg("number_of_discharges"), 0).alias("condition_avg_discharges")
    ) \
    .withColumn("condition_rank", dense_rank().over(condition_window)) \
    .withColumn("_gold_processed_at", current_timestamp())

df_condition_summary.orderBy("condition_rank").show(truncate=False)

In [0]:
# Write Gold Model 1: Hospital summary
df_hospital_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("healthcare.gold.hospital_readmission_summary")

print("Hospital summary written ✓")

# Write Gold Model 2: State summary
df_state_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("healthcare.gold.state_readmission_summary")

print("State summary written ✓")

# Write Gold Model 3: Condition summary (the one that ran earlier)
df_condition_summary = df_reportable \
    .groupBy("measure_name") \
    .agg(
        count("facility_id").alias("hospitals_reporting"),
        round(avg("excess_readmission_ratio"), 4).alias("condition_avg_excess_ratio"),
        round(avg("predicted_readmission_rate"), 4).alias("condition_avg_predicted_rate"),
        round(avg("number_of_readmissions"), 0).alias("condition_avg_readmissions"),
        round(avg("number_of_discharges"), 0).alias("condition_avg_discharges")
    ) \
    .withColumn("condition_rank", 
        dense_rank().over(Window.orderBy(col("condition_avg_excess_ratio").desc()))) \
    .withColumn("_gold_processed_at", current_timestamp())

df_condition_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("healthcare.gold.condition_readmission_summary")

print("Condition summary written ✓")

In [0]:
# Verify all three Gold tables
print("=== Gold Layer Verification ===\n")

print("Hospital summary:")
spark.sql("SELECT COUNT(*) as hospitals FROM healthcare.gold.hospital_readmission_summary").show()

print("State summary:")
spark.sql("SELECT COUNT(*) as states FROM healthcare.gold.state_readmission_summary").show()

print("Condition summary:")
spark.sql("SELECT COUNT(*) as conditions FROM healthcare.gold.condition_readmission_summary").show()

print("Top 3 penalised hospitals nationally:")
spark.sql("""
    SELECT facility_name, state, avg_excess_ratio, risk_tier, national_rank
    FROM healthcare.gold.hospital_readmission_summary
    WHERE penalty_flag = true
    ORDER BY national_rank
    LIMIT 3
""").show(truncate=False)